# Tamil TTS POC: GitHub → Colab → Google Drive

This notebook:
1. Pulls code from GitHub
2. Installs dependencies (indic-parler-tts)
3. Generates Tamil speech using Jaya's voice
4. Zips output and pushes to Google Drive

**Important:** Change runtime to **GPU** (Runtime → Change runtime type → T4 GPU)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone the GitHub Repository

In [ ]:
import os

os.chdir("/content")

GITHUB_REPO_URL = "https://github.com/saravmani-kmu/poc_colab.git"
BRANCH = "main"
REPO_DIR = "/content/poc_colab"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone --branch {BRANCH} --single-branch {GITHUB_REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
!ls -la

## Step 3: Install Dependencies

In [ ]:
!pip install transformers==4.46.1 soundfile sentencepiece "protobuf>=5.26.1,<6.0"
!pip install git+https://github.com/huggingface/parler-tts.git --no-deps
!pip install descript-audio-codec descript-audiotools --no-deps

## Step 3b: Login to HuggingFace

The indic-parler-tts model is gated. You must:
1. Go to https://huggingface.co/ai4bharat/indic-parler-tts and click "Agree and access"
2. Get your token from https://huggingface.co/settings/tokens
3. Run the cell below and paste your token

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 4: Run the TTS Pipeline

In [ ]:
!python main.py

## Step 5: Play the Generated Audio

In [ ]:
from IPython.display import Audio, display
import os

output_dir = "output"
for f in sorted(os.listdir(output_dir)):
    if f.endswith(".wav"):
        print(f"\n--- {f} ---")
        display(Audio(os.path.join(output_dir, f), autoplay=False))

## Step 6: Copy Zip to Google Drive

In [ ]:
import shutil
from datetime import datetime

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/colab_poc_output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
src = "output_bundle.zip"
dst = os.path.join(DRIVE_OUTPUT_DIR, f"tts_output_{timestamp}.zip")

shutil.copy2(src, dst)
print(f"Uploaded to Google Drive: {dst}")
print(f"File size: {os.path.getsize(dst)} bytes")

## Step 7: Verify

In [ ]:
print("Files in Google Drive output folder:")
for f in sorted(os.listdir(DRIVE_OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_OUTPUT_DIR, f))
    print(f"  {f} ({size} bytes)")